# Add noise to a Stim circuit

qLDPC noise models take a noiseless `stim.Circuit` and return a circuit with explicit error instructions.
This notebook starts with a small circuit, compares built-in and custom noise, and then samples syndromes and detectors from a surface-code memory experiment.


## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.


In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes
from qldpc.objects import Pauli

## A small circuit

The circuit prepares the three-qubit GHZ state $(|000\rangle+|111\rangle)/\sqrt{2}$ and measures every qubit in the Z basis.
Keeping the circuit small makes the inserted noise easy to inspect.


In [3]:
circuit = stim.Circuit("""
    R 0 1 2
    H 0
    CX 0 1 0 2
    M 0 1 2
""")
print(circuit)

R 0 1 2
H 0
CX 0 1 0 2
M 0 1 2


## Uniform depolarizing noise

`DepolarizingNoiseModel(p)` uses the same parameter for every supported operation.
A one-qubit Clifford is followed by a uniformly random nonidentity one-qubit Pauli with total probability `p`; a two-qubit Clifford uses the corresponding 15-element two-qubit channel.
Measurements report the wrong outcome with probability `p`, and resets prepare the orthogonal state with probability `p`.

Idle noise is optional because whether and how idle qubits fail depends on the hardware model.


In [4]:
p = 1e-3
no_idle_noise = circuits.DepolarizingNoiseModel(p, include_idling_error=False)
with_idle_noise = circuits.DepolarizingNoiseModel(p, include_idling_error=True)

print("without idle noise:")
print(no_idle_noise.noisy_circuit(circuit))
print()
print("with idle noise:")
print(with_idle_noise.noisy_circuit(circuit))

without idle noise:
R 0 1 2
X_ERROR(0.001) 0 1 2
TICK
H 0
DEPOLARIZE1(0.001) 0
TICK
CX 0 1
DEPOLARIZE2(0.001) 0 1
TICK
CX 0 2
DEPOLARIZE2(0.001) 0 2
TICK
M(0.001) 0 1 2

with idle noise:
R 0 1 2
X_ERROR(0.001) 0 1 2
TICK
H 0
DEPOLARIZE1(0.001) 0 1 2
TICK
CX 0 1
DEPOLARIZE2(0.001) 0 1
DEPOLARIZE1(0.001) 2
TICK
CX 0 2
DEPOLARIZE2(0.001) 0 2
DEPOLARIZE1(0.001) 1
TICK
M(0.001) 0 1 2


## Set operation-specific rates

Use `NoiseModel` when one- and two-qubit gates, measurements, resets, and idle moments need different rates.
The model also accepts explicit per-gate rules for hardware-specific channels; see the `NoiseModel` API page for those forms.


In [5]:
custom_noise = circuits.NoiseModel(
    clifford_1q_error=0.002,
    clifford_2q_error=0.02,
    readout_error=0.05,
    reset_error=0.01,
    idle_error=0.0001,
)
custom_noisy_circuit = custom_noise.noisy_circuit(circuit)
print(custom_noisy_circuit)

R 0 1 2
X_ERROR(0.01) 0 1 2
TICK
H 0
DEPOLARIZE1(0.002) 0
DEPOLARIZE1(0.0001) 1 2
TICK
CX 0 1
DEPOLARIZE2(0.02) 0 1
DEPOLARIZE1(0.0001) 2
TICK
CX 0 2
DEPOLARIZE2(0.02) 0 2
DEPOLARIZE1(0.0001) 1
TICK
M(0.05) 0 1 2


## Vary noise by qubit or coupler

The rates above depend only on the gate type.
For calibration data that varies across a device, `rule_func` can inspect each gate and its target qubits before choosing a `NoiseRule`.
qLDPC calls the function once per gate application, even when one Stim instruction contains several one- or two-qubit gates.
Returning `None` uses the model's ordinary defaults.

This example makes the coupling between qubits 0 and 1 noisier than the coupling between qubits 0 and 2.


In [6]:
def noisy_coupler_rule(op: stim.CircuitInstruction) -> circuits.NoiseRule | None:
    qubits = {target.qubit_value for target in op.targets_copy() if not target.is_combiner}
    if op.name == "CX" and qubits == {0, 1}:
        return circuits.NoiseRule(after=circuits.PauliChannel.depolarizing(2, 0.05))
    return None


location_dependent_noise = circuits.NoiseModel(
    clifford_1q_error=0.001,
    clifford_2q_error=0.01,
    readout_error=0.02,
    reset_error=0.01,
    rule_func=noisy_coupler_rule,
)
print(location_dependent_noise.noisy_circuit(circuit))

R 0 1 2
X_ERROR(0.01) 0 1 2
TICK
H 0
DEPOLARIZE1(0.001) 0
TICK
CX 0 1
DEPOLARIZE2(0.05) 0 1
TICK
CX 0 2
DEPOLARIZE2(0.01) 0 2
TICK
M(0.02) 0 1 2


`NoiseModel.noisy_circuit` treats operations between `TICK` instructions as one moment.
By default it can insert ticks when a qubit is reused, so parallel operations remain parallel and idle noise is applied once per moment.
Pass `insert_ticks=False` when the input circuit already contains the timing boundaries you want.


In [7]:
timed_circuit = stim.Circuit("""
    H 0
    TICK
    H 1
    TICK
    H 2
""")
print(with_idle_noise.noisy_circuit(timed_circuit, insert_ticks=False))

H 0
DEPOLARIZE1(0.001) 0 1 2
TICK
H 1
DEPOLARIZE1(0.001) 1 0 2
TICK
H 2
DEPOLARIZE1(0.001) 2 0 1


## A noisy surface-code memory experiment

`get_memory_experiment` prepares a logical eigenstate, measures stabilizers for several rounds, and reads out the logical state.
Passing the noise model during construction lets qLDPC account for moments across the initialization, repeated QEC, and final readout.

Here we use `SI1000NoiseModel`, the superconducting-inspired model from [A Fault-Tolerant Honeycomb Memory](https://arxiv.org/abs/2108.10457).
Its parameter `p` is the two-qubit gate error rate; the model sets one-qubit gates and ordinary idling to `p / 10`, resets to `2 * p`, readout to `5 * p`, and extra waiting noise during measurements and resets to `2 * p`.


In [8]:
code = codes.SurfaceCode(3)
num_rounds = 3
memory_noise = circuits.SI1000NoiseModel(0.01)
noisy_memory = circuits.get_memory_experiment(
    code,
    basis=Pauli.Z,
    num_rounds=num_rounds,
    noise_model=memory_noise,
)

print("code parameters:", code.get_code_params())
print("circuit qubits:", noisy_memory.num_qubits)
print("measurements:", noisy_memory.num_measurements)
print("detectors:", noisy_memory.num_detectors)
print("observables:", noisy_memory.num_observables)

code parameters: (9, 1, 3)
circuit qubits: 17
measurements: 33
detectors: 16
observables: 1


Syndromes are raw stabilizer outcomes.
For this fixed-basis circuit, qLDPC measures all X- and Z-type checks internally, but the final data readout closes only the Z-type detector boundaries needed to protect the logical Z observable.


In [9]:
measurement_sampler = noisy_memory.compile_sampler(seed=1234)
measurements = measurement_sampler.sample(shots=1)[0]

syndromes = measurements[: -len(code)].reshape(num_rounds, 2, code.num_checks // 2)
syndromes_z = syndromes[:, 1, :]


def bitmask(bits):
    return "".join("1" if bit else "_" for bit in bits)


print("Z-check outcomes by round:")
for syndrome in syndromes_z:
    print(bitmask(syndrome))

Z-check outcomes by round:
_1__
___1
____


Detectors compare outcomes whose parity is deterministic in the absence of faults.
Most compare neighboring syndrome rounds, while boundary detectors compare the first round with the prepared state or the last round with the final data measurements.
Those boundary checks are why this circuit has one more detector time slice than syndrome-measurement rounds.


In [10]:
detector_sampler = noisy_memory.compile_detector_sampler(seed=1234)
detectors = detector_sampler.sample(shots=1)[0]
detectors_by_time = detectors.reshape(num_rounds + 1, code.num_checks_z)

print("detector events by time slice:")
for events in detectors_by_time:
    print(bitmask(events))

detector events by time slice:
_1__
_1_1
___1
____


The logical-error-rate notebooks take the next step: they hand noisy memory circuits to Sinter, configure a decoder, and estimate failure rates over many shots.
